In [ ]:
from langgraph.graph import StateGraph, START , END
# from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import MemorySaver

# calling model
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key="your API key"
)
print(model)

metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}} profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x00000200E1D9AA50> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000200E1D9B4D0> model_name='openai/gpt-oss-120b' temperature=1e-08 model_kwargs={} groq_api_key=SecretStr('**********')


In [ ]:
from langgraph.graph import add_messages

def ChatState(TypedDict): 

     messages : Annotated[list[BaseMessage], add_messages]
    
    # add_messages --> reducer to keep storing prev msg in the list, import for retaining contexts

    # BaseMessage --> type of msg

    
    


In [ ]:
def chat_node(state : ChatState):

    # take user message from state
    messages = state['messages']  

    #send to llm 
    response = model.invoke(messages)

    #response to store in state
    return {'messages' : [response]}



In [28]:
checkpointer = MemorySaver()

# graph network
graph = StateGraph(ChatState)

# nodes 
graph.add_node('llm_node', chat_node)

# edges
graph.add_edge(START, 'llm_node')
graph.add_edge( 'llm_node', END)

#compile 

chatbot = graph.compile()

In [ ]:
input_query = {
         'messages' : [HumanMessage(content = 'what is the capital of Mexico')]
}

Final_output = chatbot.invoke(input_query)['messages'][-1].content

print(Final_output)

The capital of Mexico is **Mexico City**.


In [26]:
thread_id = '1'
while True:

    user_message = input('Type here: ')

    print("user :", user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable' : {'thread_id': thread_id } }
    response = chatbot.invoke({'messages' : [HumanMessage(content = user_message)]}, config = config)

    print('AI:', response['messages'][-1].content)
    # TypeError: 'str' object is not callable

user : what
AI: I’m not sure what you’re asking about—could you please give me a bit more detail or clarify your question? I’ll be happy to help!
user : what is the capital of UK
AI: The capital of the United Kingdom (UK) is **London**.
user : sum of 100 and 100
AI: The sum of 100 and 100 is **200**.
user : result + 20
AI: I’m not sure which result you’d like to add 20 to. Could you let me know the number (or the previous calculation) you’re referring to? Then I can give you the updated result.
user : 
AI: Hello! How can I assist you today?
user : my name is Aditi
AI: Nice to meet you, Aditi! How can I assist you today?
user : what is my name?
AI: I don’t have any information about your name. If you’d like, you can tell me what you’d like to be called!
user : 
AI: Hello! How can I assist you today?
user : 
AI: Hello! How can I assist you today?
user : 
AI: Hello! How can I assist you today?
user : 
AI: Hello! How can I assist you today?
user : 
AI: Hello! How can I assist you today?
us

In [29]:
chatbot.get_state(config = config)

NameError: name 'config' is not defined